# FindLaw — California Insurance Code → PDFs

**Prefer the official source:** open **`ca_insurance_code_official.ipynb`** first — it pulls from **`leginfo.legislature.ca.gov`** (no FindLaw/Cloudflare) and saves **Markdown** your RAG already indexes.

---

Starting from **[California Insurance Code on FindLaw](https://codes.findlaw.com/ca/insurance-code/)**, this notebook:

1. Opens pages in **Chromium (Playwright)** — FindLaw sits behind **Cloudflare**, so plain `httpx`/`requests` often get **403**.
2. Collects **same-site links** under `https://codes.findlaw.com/ca/insurance-code/`.
3. Saves each page as a **PDF** under **`data/california/ins_codes/`**.

**Legal / ethical:** Respect [FindLaw’s terms](https://www.findlaw.com/legal/terms-of-use) and `robots.txt`. Use reasonable delays. This is for **personal / research** archiving; not for republishing their content at scale.

**If you see empty PDFs or timeouts:** set `HEADLESS = False` in the config cell, run once, complete any browser challenge manually, then re-run. You can also raise `PAGE_GOTO_TIMEOUT_MS`.

## 1) Install dependencies (run once per environment)

Requires **Playwright** + **Chromium**.

In [12]:
%pip install -q playwright beautifulsoup4

You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [13]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)

CompletedProcess(args=['/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python', '-m', 'playwright', 'install', 'chromium'], returncode=0)

## 2) Configuration

In [14]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urldefrag, urljoin, urlparse

# --- URLs ---
BASE_PAGE = "https://codes.findlaw.com/ca/insurance-code/"
URL_PREFIX = "https://codes.findlaw.com/ca/insurance-code"  # crawl only under this path

# --- Output ---
OUT_DIR = Path("data") / "california" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Crawl limits (Insurance Code has many section pages; adjust as needed) ---
MAX_DISCOVER_PAGES = 4000   # max HTML pages opened while collecting links
MAX_PDF_URLS = 5000         # max distinct URLs to export as PDF (None = no cap)
DELAY_SEC = 1.0           # pause between navigations (be polite)

# --- Playwright ---
HEADLESS = True
PAGE_GOTO_TIMEOUT_MS = 90_000
VIEWPORT = {"width": 1280, "height": 2000}

USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
)

## 3) Helpers: normalize URL → safe filename

In [15]:
def under_prefix(url: str) -> bool:
    u, _ = urldefrag(url.strip())
    u = u.rstrip("/") or u
    p = URL_PREFIX.rstrip("/")
    return u == p or u.startswith(p + "/")


def normalize_url(url: str) -> str | None:
    url, _frag = urldefrag(url.strip())
    parts = urlparse(urljoin(BASE_PAGE, url))
    if parts.scheme not in ("http", "https"):
        return None
    host = (parts.hostname or "").lower()
    # FindLaw often uses www in <a href> even when the browser shows codes.findlaw.com
    if host not in ("codes.findlaw.com", "www.codes.findlaw.com"):
        return None
    path = parts.path or "/"
    if path != "/ca/insurance-code" and not path.startswith("/ca/insurance-code/"):
        return None
    clean = f"https://codes.findlaw.com{path}"
    if not under_prefix(clean):
        return None
    return clean.rstrip("/") or clean


def urls_from_html_snippets(html: str) -> set[str]:
    """Catch section URLs embedded in HTML/JSON (e.g. ins-sect-23) that BeautifulSoup misses."""
    found: set[str] = set()
    for m in re.finditer(
        r"https://(?:www\.)?codes\.findlaw\.com(/ca/insurance-code/[a-zA-Z0-9._/-]+)",
        html,
        re.I,
    ):
        nu = normalize_url("https://codes.findlaw.com" + m.group(1).rstrip("/"))
        if nu:
            found.add(nu)
    for m in re.finditer(r'["\'](/ca/insurance-code/[a-zA-Z0-9._/-]+)["\']', html, re.I):
        nu = normalize_url(urljoin("https://codes.findlaw.com/", m.group(1).rstrip("/")))
        if nu:
            found.add(nu)
    return found


def url_to_stem(url: str) -> str:
    """Stable file stem (no .pdf) under OUT_DIR."""
    n = normalize_url(url)
    assert n
    tail = n[len(URL_PREFIX.rstrip("/")) :].lstrip("/") or "index"
    stem = re.sub(r"[^a-zA-Z0-9._-]+", "_", tail.replace("/", "__"))
    stem = re.sub(r"_+", "_", stem).strip("._") or "page"
    return stem[:180]


def pdf_path_for(url: str) -> Path:
    return OUT_DIR / f"{url_to_stem(url)}.pdf"

## 4) Discover all in-scope links (BFS with Playwright)

FindLaw section pages look like `…/ca/insurance-code/ins-sect-23/`. Discovery uses **three** sources so we do not miss them:

1. **BeautifulSoup** on the final HTML  
2. **Regex** on raw HTML (catches URLs inside JSON/scripts)  
3. **`page.evaluate`** — every `<a href>` resolved in the **live DOM** (catches JS-rendered TOC)

Also: many `<a href>` values use **`www.codes.findlaw.com`** — the normalizer now maps those to `codes.findlaw.com`. The page is **scrolled** to trigger lazy-loaded section lists.

**Jupyter:** use the **async** API (`await`) here.

In [16]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup


def extract_links(html: str, page_url: str) -> set[str]:
    out: set[str] = set()
    soup = BeautifulSoup(html, "html.parser")
    for a in soup.find_all("a", href=True):
        nu = normalize_url(urljoin(page_url, a["href"]))
        if nu:
            out.add(nu)
    return out


async def js_collect_hrefs(page) -> set[str]:
    """All resolved <a href> from the live DOM (catches JS-rendered TOC)."""
    hrefs = await page.evaluate(
        """() => {
            const s = new Set();
            for (const a of document.querySelectorAll('a[href]')) {
                try { s.add(new URL(a.getAttribute('href'), document.baseURI).href); } catch (e) {}
            }
            return [...s];
        }"""
    )
    out: set[str] = set()
    for h in hrefs:
        nu = normalize_url(h)
        if nu:
            out.add(nu)
    return out


async def scroll_to_load_lazy(page) -> None:
    """Many code sites lazy-load the section list; wheel past the fold."""
    for _ in range(25):
        await page.mouse.wheel(0, 900)
        await asyncio.sleep(0.15)


async def discover_urls() -> list[str]:
    seen: set[str] = set()
    queue: list[str] = []
    start = normalize_url(BASE_PAGE)
    assert start
    queue.append(start)
    seen.add(start)

    pages_opened = 0
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(user_agent=USER_AGENT, viewport=VIEWPORT)
        page = await context.new_page()
        while queue and pages_opened < MAX_DISCOVER_PAGES:
            u = queue.pop(0)
            print(f"discover [{pages_opened + 1}] {u}", flush=True)
            try:
                await page.goto(u, wait_until="domcontentloaded", timeout=PAGE_GOTO_TIMEOUT_MS)
                try:
                    await page.wait_for_load_state("networkidle", timeout=25_000)
                except Exception:
                    pass
                await asyncio.sleep(2.0)
                await scroll_to_load_lazy(page)
                await asyncio.sleep(1.0)
                html = await page.content()
            except Exception as e:
                print(f"  skip (load error): {e}", flush=True)
                await asyncio.sleep(DELAY_SEC)
                continue
            pages_opened += 1

            merged: set[str] = set()
            merged |= extract_links(html, u)
            merged |= urls_from_html_snippets(html)
            merged |= await js_collect_hrefs(page)

            for link in merged:
                if link not in seen:
                    seen.add(link)
                    queue.append(link)
            print(f"  → queue+{len(merged)} links (unique total {len(seen)})", flush=True)
            await asyncio.sleep(DELAY_SEC)
        await browser.close()

    ordered = sorted(seen)
    if MAX_PDF_URLS is not None:
        ordered = ordered[:MAX_PDF_URLS]
    return ordered


urls = await discover_urls()
print(f"Total URLs to PDF: {len(urls)}")
(OUT_DIR / "_url_list.txt").write_text("\n".join(urls), encoding="utf-8")

discover [1] https://codes.findlaw.com/ca/insurance-code
  → queue+1 links (unique total 1)
Total URLs to PDF: 1


43

## 5) Save each page as PDF

Skips URLs whose PDF already exists (delete a `.pdf` to force regenerate).

In [17]:
import asyncio
from playwright.async_api import async_playwright


async def export_pdfs(urls: list[str]) -> None:
    ok, skip, fail = 0, 0, 0
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(user_agent=USER_AGENT, viewport=VIEWPORT)
        page = await context.new_page()
        for i, u in enumerate(urls, 1):
            dest = pdf_path_for(u)
            if dest.exists() and dest.stat().st_size > 500:
                skip += 1
                continue
            print(f"[{i}/{len(urls)}] PDF → {dest.name}", flush=True)
            try:
                await page.goto(u, wait_until="domcontentloaded", timeout=PAGE_GOTO_TIMEOUT_MS)
                await asyncio.sleep(1.2)
                dest.parent.mkdir(parents=True, exist_ok=True)
                await page.pdf(
                    path=str(dest),
                    format="Letter",
                    print_background=True,
                    margin={"top": "12mm", "bottom": "12mm", "left": "10mm", "right": "10mm"},
                )
                ok += 1
            except Exception as e:
                print(f"  FAIL: {e}", flush=True)
                fail += 1
            await asyncio.sleep(DELAY_SEC)
        await browser.close()
    print(f"Done. wrote={ok} skipped_existing={skip} failed={fail}")


await export_pdfs(urls)

Done. wrote=0 skipped_existing=1 failed=0


## Notes

- **Resume:** re-run the PDF cell; existing non-tiny PDFs are skipped.
- **Full corpus:** raise `MAX_DISCOVER_PAGES` / `MAX_PDF_URLS` after a test run.
- **403 / challenge:** use `HEADLESS = False`, complete the check in the window, then run again; or try Playwright’s `channel="chrome"` if you have Google Chrome installed (`launch(channel="chrome")`).
- **RAG ingest:** after PDFs land in `data/california/ins_codes/`, re-run your app ingest so `data/` is indexed (your default `PDFS_DIRS` / `data` root already includes this folder).